# Disneyland Reviews: EDA & Actionable Insights

Exploratory Data Analysis to uncover patterns and actionable insights for improving customer experience.
This notebook analyzes both metadata and review text to identify key themes, pain points, and opportunities.

## Part 1: Setup & Data Loading

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from rag.config import DATA_PATH

print('Loading data...')
df = pd.read_csv(DATA_PATH, encoding='latin-1')
print(f'Loaded {len(df)} reviews')
print(f'Columns: {df.columns.tolist()}')
print(f'\nFirst few rows:')
df.head()

## Part 2: Metadata Overview & Distribution Analysis

In [ ]:
print('Data Quality:')
print(f'Total reviews: {len(df)}')
print(f'Missing values:')
print(df.isnull().sum())
print(f'\nRating distribution:')
print(df['Rating'].value_counts().sort_index())
print(f'\nBranches:')
print(df['Branch'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
rating_counts = df['Rating'].value_counts().sort_index()
ax1.bar(rating_counts.index, rating_counts.values, color='steelblue', alpha=0.7, edgecolor='black')
ax1.set_xlabel('Rating', fontsize=12, fontweight='bold')
ax1.set_ylabel('Count', fontsize=12, fontweight='bold')
ax1.set_title('Overall Rating Distribution', fontsize=13, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

ax2 = axes[1]
branch_ratings = df.groupby(['Branch', 'Rating']).size().unstack(fill_value=0)
branch_ratings.plot(kind='bar', ax=ax2, color=['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd'])
ax2.set_xlabel('Branch', fontsize=12, fontweight='bold')
ax2.set_ylabel('Count', fontsize=12, fontweight='bold')
ax2.set_title('Rating Distribution by Branch', fontsize=13, fontweight='bold')
ax2.legend(title='Rating', bbox_to_anchor=(1.05, 1), loc='upper left')
ax2.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('Average rating by branch:')
print(df.groupby('Branch')['Rating'].agg(['mean', 'std', 'count']).round(2))

In [ ]:
df['YearMonth'] = pd.to_datetime(df['Year_Month'], format='%Y-%m', errors='coerce')
df_valid_dates = df.dropna(subset=['YearMonth'])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax1 = axes[0]
reviews_by_month = df_valid_dates.groupby('YearMonth').size()
ax1.plot(reviews_by_month.index, reviews_by_month.values, marker='o', linestyle='-', linewidth=2, markersize=4)
ax1.set_xlabel('Date', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of Reviews', fontsize=12, fontweight='bold')
ax1.set_title('Review Volume Over Time', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)

ax2 = axes[1]
avg_rating_by_month = df_valid_dates.groupby('YearMonth')['Rating'].mean()
ax2.plot(avg_rating_by_month.index, avg_rating_by_month.values, marker='s', linestyle='-', linewidth=2, markersize=4, color='green')
ax2.axhline(y=df['Rating'].mean(), color='red', linestyle='--', label='Overall avg', linewidth=2)
ax2.set_xlabel('Date', fontsize=12, fontweight='bold')
ax2.set_ylabel('Average Rating', fontsize=12, fontweight='bold')
ax2.set_title('Average Rating Over Time', fontsize=13, fontweight='bold')
ax2.set_ylim([1, 5])
ax2.grid(True, alpha=0.3)
ax2.legend()
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
print('Top 15 reviewer locations:')
top_locations = df['Reviewer_Location'].value_counts().head(15)
print(top_locations)

top_10_locations = df['Reviewer_Location'].value_counts().head(10).index
location_ratings = df[df['Reviewer_Location'].isin(top_10_locations)].groupby('Reviewer_Location')['Rating'].agg(['mean', 'count']).sort_values('mean', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
location_ratings['mean'].plot(kind='barh', ax=ax, color='skyblue', edgecolor='black')
ax.set_xlabel('Average Rating', fontsize=12, fontweight='bold')
ax.set_title('Average Rating by Top Reviewer Locations', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print('Average rating by top locations:')
print(location_ratings.round(2))

In [ ]:
df['TextLength'] = df['Review_Text'].fillna('').apply(len)
df['WordCount'] = df['Review_Text'].fillna('').apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.hist(df['TextLength'], bins=50, color='coral', edgecolor='black', alpha=0.7)
ax1.set_xlabel('Character Count', fontsize=12, fontweight='bold')
ax1.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax1.set_title('Review Text Length Distribution', fontsize=13, fontweight='bold')
ax1.axvline(df['TextLength'].median(), color='red', linestyle='--', linewidth=2, label=f'Median: {df["TextLength"].median():.0f}')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

ax2 = axes[1]
ax2.hist(df['WordCount'], bins=50, color='lightgreen', edgecolor='black', alpha=0.7)
ax2.set_xlabel('Word Count', fontsize=12, fontweight='bold')
ax2.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax2.set_title('Review Word Count Distribution', fontsize=13, fontweight='bold')
ax2.axvline(df['WordCount'].median(), color='red', linestyle='--', linewidth=2, label=f'Median: {df["WordCount"].median():.0f}')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Part 3: LLM-Based Text Analysis (Strategic Sample)

In [ ]:
import os
from langchain_litellm import ChatLiteLLM
from rag.config import LLM_MODEL_NAME
import json

proxy_url = os.getenv('LITELLM_PROXY_URL', 'https://litellm.gke-prod.linnovate.net')
api_key = os.getenv('LITELLM_MASTER_KEY')

llm = ChatLiteLLM(
    model=LLM_MODEL_NAME,
    api_base=proxy_url,
    api_key=api_key,
    timeout=120,
)

np.random.seed(42)
sample_per_branch = 40
samples = []
for branch in df['Branch'].unique():
    branch_df = df[df['Branch'] == branch]
    branch_sample = branch_df.groupby('Rating', group_keys=False).apply(
        lambda x: x.sample(n=min(len(x), max(1, int(sample_per_branch * len(x) / len(branch_df)))), random_state=42)
    )
    samples.append(branch_sample)

sample_df = pd.concat(samples, ignore_index=True).drop_duplicates(subset=['Review_ID'])
print(f'Stratified sample size: {len(sample_df)} reviews')
print(f'Sample by branch:')
print(sample_df['Branch'].value_counts())

In [ ]:
def analyze_review(review_text, rating, branch, location):
    prompt = f"""Analyze this Disneyland review. Rate: {rating}/5, Branch: {branch}, Location: {location}

Review: {review_text[:500]}

Return JSON with:
- sentiment: positive, neutral, or negative
- issues: list of any problems
- highlights: list of positives

JSON only:"""
    response = llm.invoke(prompt)
    response_text = response.content.strip()
    
    json_start = response_text.find('{')
    json_end = response_text.rfind('}') + 1
    if json_start != -1 and json_end > json_start:
        response_text = response_text[json_start:json_end]
    
    try:
        return json.loads(response_text)
    except:
        return {}

print(f'Analyzing {min(100, len(sample_df))} reviews...')
analyses = []
for idx, (_, row) in enumerate(sample_df.head(100).iterrows()):
    if idx % 10 == 0:
        print(f'  {idx}/100')
    analysis = analyze_review(row['Review_Text'], row['Rating'], row['Branch'], row['Reviewer_Location'])
    analysis['review_id'] = row['Review_ID']
    analysis['rating'] = row['Rating']
    analysis['branch'] = row['Branch']
    analyses.append(analysis)

print(f'Completed {len(analyses)} analyses')

In [ ]:
analyses_df = pd.DataFrame(analyses)
print(f'Analyzed {len(analyses_df)} reviews')

if 'sentiment' in analyses_df.columns:
    print('\nSentiment distribution:')
    print(analyses_df['sentiment'].value_counts())
    
    fig, ax = plt.subplots(figsize=(8, 5))
    sentiment_counts = analyses_df['sentiment'].value_counts()
    colors = {'positive': 'green', 'neutral': 'gray', 'negative': 'red'}
    sentiment_counts.plot(kind='bar', ax=ax, color=[colors.get(x, 'blue') for x in sentiment_counts.index])
    ax.set_ylabel('Count', fontweight='bold')
    ax.set_title('Sentiment Distribution (LLM Analysis)', fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
if 'issues' in analyses_df.columns:
    all_issues = []
    for issues in analyses_df['issues']:
        if isinstance(issues, list):
            all_issues.extend([i.lower().strip() for i in issues if i])
    
    issue_counts = Counter(all_issues)
    print('\nTop 10 Issues:')
    for issue, count in issue_counts.most_common(10):
        print(f'  {issue}: {count}')
    
    fig, ax = plt.subplots(figsize=(10, 5))
    top_issues = dict(issue_counts.most_common(8))
    ax.barh(list(top_issues.keys()), list(top_issues.values()), color='coral', edgecolor='black')
    ax.set_xlabel('Frequency', fontweight='bold')
    ax.set_title('Top Pain Points', fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
if 'highlights' in analyses_df.columns:
    all_highlights = []
    for highlights in analyses_df['highlights']:
        if isinstance(highlights, list):
            all_highlights.extend([h.lower().strip() for h in highlights if h])
    
    highlight_counts = Counter(all_highlights)
    print('\nTop 10 Highlights:')
    for highlight, count in highlight_counts.most_common(10):
        print(f'  {highlight}: {count}')
    
    fig, ax = plt.subplots(figsize=(10, 5))
    top_highlights = dict(highlight_counts.most_common(8))
    ax.barh(list(top_highlights.keys()), list(top_highlights.values()), color='lightgreen', edgecolor='black')
    ax.set_xlabel('Frequency', fontweight='bold')
    ax.set_title('Top Positive Aspects', fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

## Part 4: Key Insights & Recommendations

In [ ]:
print('\nKEY FINDINGS:')
print(f'  Overall avg rating: {df["Rating"].mean():.2f}/5')
print(f'  5-star reviews: {(df["Rating"] == 5).sum():,} ({(df["Rating"] == 5).sum() / len(df) * 100:.1f}%)')
print(f'  1-star reviews: {(df["Rating"] == 1).sum():,} ({(df["Rating"] == 1).sum() / len(df) * 100:.1f}%)')
print(f'  Total reviews: {len(df):,}')
print(f'  Branches: {df["Branch"].nunique()}')
print(f'  Unique locations: {df["Reviewer_Location"].nunique()}')
print(f'\nBranch comparison:')
print(df.groupby('Branch')['Rating'].agg(['mean', 'count']).round(2))